# 🏛️ Vietnamese Legal QA — QLoRA Fine-tuning on Qwen3-4B

**Runtime:** T4 GPU (~15 GB VRAM) · High RAM · Python 3.10+

---

### 📋 What this notebook does
1. Install all training dependencies
2. Convert `train1.json` → **Qwen ChatML** format
3. Load **Qwen2.5-4B** (4-bit NF4 QLoRA)
4. Fine-tune with:
   - LoRA rank=32, alpha=64, dropout=0.05
   - 4-bit NormalFloat (NF4) + double quantization
   - 8-bit AdamW · LR=2e⁻⁵ · warmup=5 steps
   - effective batch size=128 (16 × 8 grad accum)
   - max seq=2048 · 1 epoch
5. Save adapter to Google Drive

---


In [ ]:
# ── Step 1: Install dependencies ──────────────────────────
!pip install -q --upgrade pip
!pip install -q \
    transformers>=4.46.0 \
    datasets>=3.2.0 \
    peft>=0.13.0 \
    accelerate>=1.1.0 \
    bitsandbytes>=0.44.1 \
    scipy>=1.14.0 \
    safetensors>=0.4.5 \
    pyyaml>=6.0.2 \
    tqdm

import os, sys
print("✓ Dependencies installed")

In [ ]:
# ── Step 2: Upload & prepare dataset ───────────────────────
from google.colab import files
import json, re

# Upload your train1.json (or put it in /content/)
uploaded = files.upload()
DATA_FILE = list(uploaded.keys())[0]
OUT_FILE  = "/content/train1.jsonl"

# ChatML tokens (Qwen3 style)
IM_START = "<|im_start|>"
IM_END   = "<|im_end|>"
EOT      = "<|eot_id|>"

SYSTEM = (
    "Bạn là một trợ lý pháp lý chuyên nghiệp, được huấn luyện trên dữ liệu pháp luật Việt Nam. "
    "Nhiệm vụ của bạn là trả lời câu hỏi pháp lý một cách chính xác, đầy đủ và có suy luận rõ ràng. "
    "Luôn sử dụng định dạng <answer>...</answer> để đóng khung câu trả lời cuối cùng."
)

def strip_think(text):
    return re.sub(r"</?think[^>]*>", "", text).strip()

def sample_to_chatml(sample):
    # task1
    if "legal_document" in sample:
        legal_doc  = sample.get("legal_document", "")
        sub_q      = sample.get("specific_question", "")
        question   = sample.get("question", "")
        choices    = sample.get("choices", [])
        ans_idx    = int(sample.get("answer", 0))
        reasoning  = strip_think(sample.get("reasoning", ""))
        choice_text = "\n".join(f"  {i+1}. {c}" for i, c in enumerate(choices))
        ans_text   = choices[ans_idx] if ans_idx < len(choices) else "Không"
        user_msg   = (f"Tài liệu pháp lý:\n{legal_doc}\n\n"
                      f"Câu hỏi cụ thể: {sub_q}\n\n"
                      f"Câu hỏi: {question}\n"
                      f"Các lựa chọn:\n{choice_text}\n\n"
                      f"Hãy phân tích và đưa ra câu trả lời.")
        asst_msg   = f"{reasoning}\n\n<answer>{ans_text}</answer>"

    # task2
    elif "choices" in sample:
        question   = sample.get("question", "")
        choices    = sample.get("choices", [])
        ans_idx    = int(sample.get("answer", 0))
        reasoning  = strip_think(sample.get("reasoning", ""))
        choice_text = "\n".join(f"  {chr(65+i)}. {c}" for i, c in enumerate(choices))
        ans_text   = f"{chr(65+ans_idx)}. {choices[ans_idx]}"
        user_msg   = (f"Câu hỏi: {question}\n\n"
                      f"Các lựa chọn:\n{choice_text}\n\n"
                      f"Hãy phân tích từng lựa chọn và đưa ra câu trả lời đúng.")
        asst_msg   = f"{reasoning}\n\n<answer>{ans_text}</answer>"

    # task3
    else:
        question  = sample.get("question", "")
        answer    = re.sub(r"</?answer[^>]*>", "", strip_think(sample.get("answer", ""))).strip()
        user_msg  = (f"Câu hỏi: {question}\n\n"
                     f"Hãy phân tích và đưa ra câu trả lời dựa trên các quy định pháp luật.")
        lines     = answer.split("\n")
        verdict   = next((l.strip() for l in reversed(lines)
                         if l.strip() and "tiền đề" not in l.lower()), answer[:300])
        asst_msg  = f"{answer}\n\n<answer>{verdict}</answer>"

    return (
        f"{IM_START}system\n{SYSTEM}{IM_END}\n"
        f"{IM_START}user\n{user_msg}{IM_END}\n"
        f"{IM_START}assistant\n{asst_msg}{IM_END}\n"
        f"{EOT}"
    )

with open(DATA_FILE, encoding="utf-8") as f:
    data = json.load(f)

with open(OUT_FILE, "w", encoding="utf-8") as fout:
    for task_key in ["task1", "task2", "task3"]:
        for sample in data.get(task_key, []):
            fout.write(json.dumps({"text": sample_to_chatml(sample)}, ensure_ascii=False) + "\n")

import subprocess
n_lines = int(subprocess.check_output(["wc", "-l", OUT_FILE]).decode().split()[0])
print(f"✓ Converted {n_lines} samples → {OUT_FILE}")

In [ ]:
# ── Step 3: Mount Google Drive (to save adapter) ──────────
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/legal_qa_qlora"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output dir: {OUTPUT_DIR}")

In [ ]:
# ── Step 4: Check GPU ───────────────────────────────────────
import torch
assert torch.cuda.is_available(), "Runtime GPU missing! Runtime → Change runtime type → T4"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✓ GPU: {gpu_name}  ({gpu_mem:.1f} GB)")

In [ ]:
# ── Step 5: Load model + apply QLoRA ─────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "Qwen/Qwen2.5-4B"   # or "Qwen/Qwen3-4B" when available

print("[1/3] Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("[2/3] Loading base model with 4-bit NF4 …")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
base_model.enable_input_require_grads()

print("[3/3] Applying LoRA adapters …")
lora_cfg = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# ── Step 6: Prepare dataset ────────────────────────────────
from datasets import load_dataset

MAX_SEQ = 2048

def preprocess(examples):
    out = tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ,
                    return_attention_mask=False)
    out["labels"] = out["input_ids"].copy()
    return out

raw_ds = load_dataset("json", data_files=OUT_FILE, split="train")
ds = raw_ds.map(preprocess, batched=True, remove_columns=raw_ds.column_names,
                desc="Tokenizing")
print(f"✓ {len(ds)} samples tokenized (max_len={MAX_SEQ})")

In [ ]:
# ── Step 7: TrainingArguments + Trainer ────────────────────
import math
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

BS        = 16
GRAD_ACC  = 8         # effective batch size = 128
EPOCHS    = 1
LR        = 2e-5
WARMUP    = 5
MAX_SEQ   = 2048

steps_per_epoch = math.ceil(len(ds) / (BS * GRAD_ACC))
max_steps      = steps_per_epoch * EPOCHS

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BS,
    gradient_accumulation_steps=GRAD_ACC,
    max_steps=max_steps,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="linear",
    warmup_steps=WARMUP,
    optim="adamw_torch",
    bf16=True,
    fp16=False,
    logging_steps=25,
    save_steps=steps_per_epoch,
    save_total_limit=2,
    seed=42,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=4,
    tf32=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(f"  • effective batch size : {BS * GRAD_ACC}")
print(f"  • max steps            : {max_steps}")
print(f"  • warmup steps         : {WARMUP}")
print(f"  • LoRA r=32, alpha=64")
print(f"  • sequence length      : {MAX_SEQ}")
print("\n🚀 Starting training …")

In [ ]:
trainer.train()

In [ ]:
# ── Step 8: Save adapter ────────────────────────────────────
FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"✓ Adapter saved → {FINAL_DIR}")

# Quick sanity-check: reload and run a test prompt
print("\n─── Test inference ───")
from peft import PeftModel

del trainer, model, base_model
import gc; gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(FINAL_DIR, trust_remote_code=True)
tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_cfg, device_map="auto",
    trust_remote_code=True, torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base, FINAL_DIR)
model.eval()

test_q = (
    "Theo Nghị định 168/2024/NĐ-CP, người điều khiển xe ô tô chạy quá tốc độ "
    "quy định trên 35 km/h sẽ bị áp dụng mức phạt tiền bao nhiêu?"
)
prompt = (
    f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
    f"<|im_start|>user\nCâu hỏi: {test_q}\n\n"
    f"Hãy phân tích và đưa ra câu trả lời dựa trên các quy định pháp luật Việt Nam.<|im_end|>\n"
    f"<|im_start|>assistant\n"
)
inputs = tok(prompt, return_tensors="pt").to("cuda")
out = model.generate(
    **inputs, max_new_tokens=512, temperature=0.3,
    top_p=0.9, do_sample=True, eos_token_id=tok.eos_token_id,
)
print("QUESTION:", test_q)
print("\nANSWER:", tok.decode(out[0][inputs["input_ids"].shape[1]:],
                             skip_special_tokens=False)
                 .replace("<|im_end|>", "").replace("<|eot_id|>", "").strip())